In [ ]:
# Packages and modules
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Statevector, Pauli
from qiskit_aer import AerSimulator
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from qiskit_ibm_runtime import  EstimatorV2
from qiskit.quantum_info import SparsePauliOp
from qiskit_ibm_runtime import QiskitRuntimeService, Batch, EstimatorV2 as Estimator
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

In [ ]:
"-------------------Floquet evolution operator construction-------------------"
def create_trotter_circuit(num_qubits, Jz, hz, hx, trotter_steps):

    qc = QuantumCircuit(num_qubits)
    for _ in range(trotter_steps):

        # Longitudinal Z field
        for i in range(num_qubits):
            qc.rz(2 * hz, i)

        # Even bonds: ZZ interactions
        for i in range(0, num_qubits - 1, 2):
            qc.rzz(2 * Jz, i, i + 1)

        # Odd bonds: ZZ interactions
        for i in range(1, num_qubits - 1, 2):
            qc.rzz(2 * Jz, i, i + 1)

        # Transverse X kick
        for i in range(num_qubits):
            qc.rx(2 * hx, i)

    return qc



"-------------------initial State construction-------------------"
# Function to create domain wall initial state on quantum circuit 
def domain_wall(num_qubits) -> QuantumCircuit:
    init_qc = QuantumCircuit(num_qubits)
    for i in range(num_qubits//2):
        init_qc.x(i)
    return init_qc

# Function to create Nèel(CDW) state on quantum circuit 
def neel(num_qubits) -> QuantumCircuit:
    init_qc = QuantumCircuit(num_qubits)
    for i in range(num_qubits) :
        if i%2 ==0:
            init_qc.x(i)

    return init_qc


"------------------- Operators -------------------"
def imbalance_operator(num_qubits: int) -> SparsePauliOp:
    pauli_list = []
    for i in range(num_qubits):
        p = ['I'] * num_qubits
        p[i] = 'Z'
        coeff = (-1) ** i
        pauli_list.append(("".join(p), coeff/num_qubits))
    return SparsePauliOp.from_list(pauli_list)


def magnetization(num_qubits) -> list:
    mag_ops = [
        SparsePauliOp.from_sparse_list([("Z", [q], 1.0)], num_qubits=num_qubits)
        for q in range(num_qubits)
    ]

    return mag_ops

In [ ]:
"------------------- backend -------------------"
service = QiskitRuntimeService()

backend_name = "ibm_kingston"
backend = service.backend(backend_name)


qiskit_runtime_service.__init__:WARNING:2026-10-14 19:51:47,475: Instance was not set at service instantiation. Free and trial plan instances will be prioritized. Based on the following filters: (tags: None, region: us-east, eu-de), and available plans: (open), the available account instances are: open-instance. If you need a specific instance set it explicitly either by using a saved account with a saved default instance or passing it in directly to QiskitRuntimeService(). Alternatively, pass instance='auto' or save it to your account for auto-selection without warning.
qiskit_runtime_service.backends:WARNING:2026-10-14 19:51:47,479: Using instance: open-instance, plan: open


In [ ]:
"------------------- Parameters -------------------"
num_qubits = 100
Jz = np.pi/2

# field-strength
hx = np.pi/2
hz = np.pi/2
# Time Period
T = 1

# Number of trotter steps 
max_ntrot = 50


# Pass manager
pass_manager = generate_preset_pass_manager(
    backend=backend,
    optimization_level=1,
    routing_method="sabre",
    seed_transpiler=593256,)

In [5]:
initial_circuit = neel(num_qubits)
operator = imbalance_operator(num_qubits)

imbalance_values = []
floquet_periods = []


transpiled_circuits = []
isa_observables = []

depth_data = []

for m in range(max_ntrot + 1):
    if m == 0:
        full_circuit = initial_circuit
    else:
        evolution_circuit = create_trotter_circuit(num_qubits, Jz*T, hz*T, hx, m)
        full_circuit = initial_circuit.compose(evolution_circuit)

    transpiled_qc = pass_manager.run(full_circuit)
    transpiled_circuits.append(transpiled_qc)
    isa_observables.append(operator.apply_layout(transpiled_qc.layout))

    total_depth = transpiled_qc.depth()
    twoq_depth = transpiled_qc.depth(filter_function=lambda inst: inst.operation.num_qubits == 2)

    depth_data.append([m, total_depth, twoq_depth])

df = pd.DataFrame(depth_data, columns=["Trotter Steps", "Total Depth", "Two-Qubit Depth"])
print(df.to_string(index=False))


 Trotter Steps  Total Depth  Two-Qubit Depth
             0            1                0
             1           15                4
             2           27                8
             3           39               12
             4           51               16
             5           63               20
             6           75               24
             7           87               28
             8           99               32
             9          111               36
            10          123               40
            11          135               44
            12          147               48
            13          159               52
            14          171               56
            15          183               60
            16          195               64
            17          207               68
            18          219               72
            19          231               76
            20          243               80
          

In [ ]:
# ------- SUBMIT BATCH JOB FOR SINGLE h VALUE --------
with Batch(backend=backend) as batch:
    estimator = Estimator(mode=batch)

    # ------- ERROR MITIGATION OPTIONS -------- 
    estimator.options.default_shots = 10000
    estimator.options.dynamical_decoupling.enable = True
    estimator.options.dynamical_decoupling.sequence_type = "XY4"
    estimator.options.resilience.measure_mitigation = True
    estimator.options.twirling.enable_gates = True
    estimator.options.twirling.num_randomizations = "auto"
    estimator.options.resilience.zne_mitigation = True
    estimator.options.resilience.zne.noise_factors = (1, 1.2, 1.4)
    estimator.options.resilience.zne.extrapolator = "linear"                 

    pairs = list(zip(transpiled_circuits, isa_observables))
    job = estimator.run(pairs)

# ------- PRINT JOB ID FOR h VALUES --------
print(f" Submitted job id: {job.job_id()}")
            

/var/folders/kj/w09c1chs6lnbp780_n6b4gmm0000gn/T/ipykernel_9587/1727622391.py:3: DeprecationWarning: The EstimatorV2 class is deprecated as of qiskit-ibm-runtime 0.50.0 and will be removed no sooner than 3 months after the release date. `qiskit_ibm_runtime.executor_estimator.Estimator` provides a new Estimator implementation that runs all the pre- and post-processing in the client side. The top-level `qiskit_ibm_runtime.Estimator` import will be updated to point to the client-side estimator in an upcoming release, with the legacy Estimator being available until the deprecation is completed.
  estimator = Estimator(mode=batch)


 Submitted job id: db7p03gr1c5c738l1470
